# 剪枝（结构化剪枝）学习笔记

本 notebook 只注重「思路 + 重要代码」，**不带运行输出**。每个流程按 **原理 / 重要 bash 命令 / 重要代码** 三部分整理。

代码取自 `scripts/`；实验 11 多方法对比的脚本在临时目录 `C:/Users/22565/AppData/Local/Temp/yolo_coco128_impr/`（`pruner.py`、`e2_criteria.py`、`e3_sparse.py`、`e4_iterative.py`、`e5_sfp_hrank.py`、`e6_kd.py` 等）。每个代码块标注了来源文件，可回原脚本看完整实现。

## 一、通用准备（所有实验共用，只讲一次）

### 1. 环境安装（ultralytics + torch-pruning）

**原理**

两个核心库：
- `ultralytics`：YOLO11 官方实现，负责模型加载、训练、验证。
- `torch-pruning`：结构化剪枝库，核心是 **DependencyGraph（依赖图）**。

**名词**
- **结构化剪枝 vs 非结构化剪枝**：结构化剪枝删「整条卷积输出通道」，体积/计算量/速度同时受益；非结构化只把单个权重置零，几乎不加速。本项目做的是结构化。
- **依赖图（DependencyGraph）**：记录层与层之间的通道耦合。剪掉某层 conv 的输出通道时，下游 BN 通道数、相连 conv 的输入通道、C2f 的 split/concat 都要跟着一起删，否则形状对不上会报错。`torch-pruning` 的 `build_dependency` + `get_pruning_group` 就是干这个。

本机环境：RTX 5060 Ti、PyTorch 2.11.0+cu128、torch-pruning 1.6.1。

**重要 bash 命令**

```bash
# 建虚拟环境（可选）
python -m venv .venv
# 安装依赖
pip install ultralytics torch torchvision
pip install torch-pruning
# 验证安装
python -c "import ultralytics, torch_pruning, torch; print(ultralytics.__version__, torch_pruning.__version__, torch.__version__)"
```

In [ ]:
# 科研里反复用到的几个入口（来自各脚本顶部 import）
import torch
import torch_pruning as tp  # 依赖图 / 剪枝 API / count_ops_and_params
from ultralytics import YOLO  # 模型加载、train/val 入口
from ultralytics.nn.modules import C2f, C2PSA  # YOLO11 的 CSP 模块，剪枝保护规则要用
from ultralytics.models.yolo.detect.train import (
    DetectionTrainer,
)  # 微调时绕过 get_model 重建

### 2. 数据下载与划分（coco8 / coco128 / coco2017）

**原理**

数据规模三档：
- `coco8`：8 张，只用于验证流程/环境是否跑通。
- `coco128`：128 张，小规模剪枝实验用（训练快，适合快速试方法）。
- `coco2017`：约 11.8 万训练图 + 5000 验证图，正式实验用。

**为什么 coco128 要自己划分**：coco128 本身没有官方 train/val 划分，脚本按 8:2（102 训练 / 26 验证）固定随机种子 42 切分，保证实验可复现。

**名词**：训练集（更新权重）/ 验证集（只测不练，用来评估精度、选模型）。

注意：coco2017 数据量大，放在仓库外的 `C:/Users/22565/datasets/coco`，不进 OneDrive。

**重要 bash 命令**

```bash
# coco8 / coco128：ultralytics 首次 train/val 时会自动下载
# 划分 coco128（生成 datasets/coco128_split）
python scripts/split_coco128.py

# coco2017：见 configs/coco2017.yaml 里的 download 段，或手动下载到 C:/Users/22565/datasets/coco
```

In [ ]:
# 来源：scripts/split_coco128.py —— 固定种子 8:2 切分
import random
from pathlib import Path

random.seed(42)
images = sorted(Path("datasets/coco128/images/train2017").glob("*.jpg"))
random.shuffle(images)
cut = int(len(images) * 0.8)  # 8:2
train, val = images[:cut], images[cut:]

### 3. 模型下载与加载（yolo11n.pt / yolo11s.pt）

**原理**

- **YOLO11n / YOLO11s**：n = nano（约 2.6M 参数），s = small（约 9.5M 参数）。小模型冗余本来就少，剪枝收益有限——这是本项目「剪得少、掉点多」的根本原因。
- **.pt 里有什么**：ultralytics 的 `.pt` 是一个 dict，含 `model`（网络结构+权重）、`ema`、`optimizer`、`train_args` 等；`YOLO(path)` 负责解析成可用的检测模型。
- **预训练权重**：在 COCO 上已训练好的权重，作为微调/剪枝的起点，比从零训练快很多。

**重要 bash 命令**

```bash
# 官方预训练权重放到 weights/ 或 models/，ultralytics 也能自动下载
# 也可用脚本触发下载：
python -c "from ultralytics import YOLO; YOLO('yolo11n.pt')"
```

In [ ]:
# 来源：scripts/run_yolo11s_coco2017_baseline.py —— 下载 + 加载
from ultralytics import YOLO
from ultralytics.utils.downloads import attempt_download_asset

weights = Path("weights/yolo11s.pt")
downloaded = Path(attempt_download_asset(weights))  # 不存在则自动下载并返回路径
model = YOLO(str(downloaded))  # 解析 .pt，拿到检测模型

### 4. 训练与监控进度（train 参数、results.csv / png、best.pt）

**原理**

核心 `train` 参数：
- `epochs`：训练轮数。
- `imgsz`：输入尺寸（常见 640，本项目 coco2017 训练用 512）。
- `batch`：批大小。
- `device`：用哪张卡（0 = 第一张 GPU）。
- `seed`：随机种子，固定后可复现。
- `patience`：早停——连续 N 个 epoch 无提升就提前停。

**best.pt 怎么来**：ultralytics 每个 epoch 在验证集上算 `fitness`（综合 mAP/recall 的分数），最高那轮存为 `best.pt`，最后一轮存 `last.pt`。

**results.csv / results.png**：训练曲线，记录每个 epoch 的 loss 和指标，用来判断有没有过拟合/训练是否正常。

**重要 bash 命令**

```bash
# 命令行方式（coco128 上微调 YOLO11n）
yolo detect train data=configs/coco128_split.yaml model=models/yolo11n.pt epochs=3 imgsz=640 batch=8 device=0
# 或跑仓库脚本（coco2017 全量训练）
python scripts/train_yolo11s_coco2017.py
```

In [ ]:
# 来源：scripts/train_yolo11s_coco2017.py —— 关键 train 设置
from ultralytics import YOLO

model = YOLO("weights/yolo11s.pt")
model.train(
    data="configs/coco2017.yaml",
    epochs=30,
    imgsz=512,
    batch=64,
    device=0,
    workers=2,
    seed=42,
    deterministic=False,
    patience=10,
    project="runs/train/xxx",
    name="时间戳",
    exist_ok=True,
    plots=True,
)

### 5. 结果指标含义（P / R / mAP50 / mAP50-95 / GMACs / 参数 / FLOPs）

**原理**（老师必问，重点）

- **Precision / Recall**：精确率 = 预测出的框里有多少是真框；召回率 = 真框里有多少被找到。二者是 trade-off。
- **mAP50 / mAP50-95**：在 IoU 阈值 0.5（或 0.5~0.95 取平均）下算的平均精度。mAP50 宽松，**mAP50-95 更严格、是主指标**。
- **参数量 (params)**：所有权重个数，影响体积和显存。
- **GMACs vs FLOPs**：都是计算量单位。**1 GMAC = 2 FLOPs**（一次乘加 MAC 算 2 次浮点运算）。仓库统一用 GMACs。
- **为什么 FLOPs 降了但速度不一定快**：减的量太少（本项目只有 ~1–4%）时，测速波动比收益还大；且真实推理还受内存带宽、算子调度影响，FLOPs 只是近似。

**重要 bash 命令**

```bash
# 验证并输出指标（用 yolo 命令行）
yolo detect val data=configs/coco128_split.yaml model=runs/train/experiment02_coco128/weights/best.pt imgsz=640 device=0
```

In [ ]:
# 取验证指标（来源：scripts/run_yolo11s_coco2017_baseline.py）
metrics = model.val(data=..., split="val", imgsz=640, batch=32, device=0)
map50 = float(metrics.box.map50)  # mAP50
map50_95 = float(metrics.box.map)  # mAP50-95
precision = float(metrics.box.mp)  # Precision
recall = float(metrics.box.mr)  # Recall
infer_ms = float(metrics.speed["inference"])  # 每张图验证推理耗时(ms)

# 统计 GMACs 和参数量（来源：scripts/prune_independent_compare.py）
import torch_pruning as tp

macs, _ = tp.utils.count_ops_and_params(
    model, example_inputs=torch.zeros(1, 3, 640, 640)
)
gmacs = macs / 1e9  # 除以 1e9 得到 GMACs
params = sum(p.numel() for p in model.parameters())

## 二、各实验专属流程

### 敏感度分析（实验 03 coco128 / 实验 09 coco2017）

**原理**

- **目的**：找出哪些层的通道「剪了不疼」，为后续结构化剪枝筛候选层。
- **方法（masking 置零）**：每次从同一个 best.pt 重载，把某一层 L1 重要性最低的约 10% 输出通道权重**置零（不是真删）**，在验证集上测精度下降量 `drop`。
- **L1 重要性**：卷积核权重绝对值越大越重要，用 `weight.abs().mean()` 给每个输出通道打分。
- **关键区别**：masking 只是置零估趋势，**不减少参数量/GMACs**，也不等价于真剪枝（真剪会删通道、改结构）。所以它是「启发式筛选」，不是精确预测。

**名词**：敏感度 = 屏蔽该层后 mAP 下降越多，越敏感、越不能剪。

**重要 bash 命令**

```bash
python scripts/prune_sensitivity.py \
    --weights runs/train/experiment02_coco128/weights/best.pt \
    --data configs/coco128_split.yaml --ratio 0.10 \
    --output reports/experiment03_sensitivity.csv
```

In [ ]:
# 来源：scripts/prune_sensitivity.py —— 候选层 + L1 置零
def find_candidate_layers(model):
    # 全部输出通道 >= 16 的卷积层
    return [
        (n, m)
        for n, m in model.model.named_modules()
        if isinstance(m, torch.nn.Conv2d) and m.out_channels >= 16
    ]


def mask_low_l1_filters(conv, ratio):
    count = max(1, round(conv.out_channels * ratio))  # 要屏蔽的通道数
    count = min(count, conv.out_channels - 1)
    scores = (
        conv.weight.detach().abs().flatten(1).mean(1)
    )  # L1：每个输出通道权重绝对值均值
    indices = torch.argsort(scores)[:count]  # 取最小的一批
    with torch.no_grad():
        conv.weight[indices] = 0  # 置零（不是删除）
        if conv.bias is not None:
            conv.bias[indices] = 0
    return count

### 独立法结构化剪枝（实验 04）

**原理**

- **独立法**：light/balanced/strong 三组分别从**同一个 best.pt 独立**开始，剪不同数量的低敏感层（3/6/9 个），互不影响。
- **真正的结构化剪枝**：这次真的删通道——`DependencyGraph` 自动把下游 BN、相连 conv 输入通道、深度卷积宽度一起删，保证形状对齐。
- **候选层筛选（read_candidates）**：从敏感度 CSV 里挑「低敏感（mAP50-95 drop ≤ 0.005）+ 输出通道 ≥ 64 + 排除首层/注意力/检测头」的层。
- **choose_indices**：按 L1 打分选要删的通道，并把数量**对齐到 8 的倍数**（channel_multiple=8），硬件对 8 的倍数通道更友好。
- **微调**：剪完用 `finetune_pruned_compare.py` 恢复精度（见贪心块里的微调代码）。

**重要 bash 命令**

```bash
python scripts/prune_independent_compare.py \
    --weights runs/train/experiment02_coco128/weights/best.pt \
    --data configs/coco128_split.yaml \
    --sensitivity reports/experiment03_sensitivity.csv \
    --profiles light balanced strong
```

In [ ]:
# 来源：scripts/prune_independent_compare.py —— 选通道 + 真剪核心
def choose_indices(conv, ratio, channel_multiple):
    desired = max(1, round(conv.out_channels * ratio))
    if conv.out_channels >= channel_multiple * 2:  # 对齐到 8 的倍数
        desired = max(
            channel_multiple, round(desired / channel_multiple) * channel_multiple
        )
        desired = min(desired, conv.out_channels - channel_multiple)
    scores = conv.weight.detach().abs().flatten(1).mean(1)  # L1 打分
    return torch.argsort(scores)[:desired].cpu().tolist()  # 最低的 desired 个


def prune_one_layer(model, layer_name, ratio, channel_multiple, example):
    for p in model.parameters():
        p.requires_grad_(True)  # 关键：ultralytics 加载后 requires_grad=False，
        # 不设 True 依赖图会追踪不到可剪模块
    conv = dict(model.named_modules())[layer_name]
    indices = choose_indices(conv, ratio, channel_multiple)
    graph = tp.DependencyGraph().build_dependency(
        model, example_inputs=example
    )  # 建依赖图
    group = graph.get_pruning_group(
        conv, tp.prune_conv_out_channels, idxs=indices
    )  # 拿到联动组
    if not graph.check_pruning_group(group):
        raise RuntimeError("依赖图拒绝剪枝")
    group.prune()  # 一次性删掉 conv + 下游所有联动通道

### 贪心结构化剪枝（实验 05 / 10）

**原理**

- **贪心搜索**：每一步在所有候选层上「试剪」，选**代价最小**的一步接受。代价 = mAP 下降 / GMAC 减少（每省一点计算量掉多少精度）。每步都在当前已剪模型上重新算 L1、重新选，允许同一层重复剪。
- **与独立法的区别**：独立法一次性剪多个层；贪心一步一步来，每步只接受当前最优。
- **保护规则（safe_prune）**：不剪首层(stem)、注意力层、C2f/C2PSA 的 CSP 分块宽度、检测头最终输出（DFL/类别框语义）；每层至少保留初始通道一半。
- **微调恢复（run_finetune）**：剪完精度掉，用少量轮次微调补回来。两个关键点：① 直接 `trainer.model = pruned.model`，绕过 ultralytics 按 YAML 重建（否则剪掉的结构会被重建回去）；② 固定 BN（统计量和仿射参数都不更新），因为小数据会把 BN 统计冲坏。

**名词**：backward_check = 剪完做前向+反向检查，确认梯度有限、结构没坏才接受这一步。

**重要 bash 命令**

```bash
# coco128 贪心
python scripts/prune_greedy.py --device 0 --max-steps 20
# coco2017 贪心
python scripts/prune_greedy_coco2017.py --device 0 --max-steps 20 --target-reduction 0.20
```

In [ ]:
# 来源：scripts/prune_greedy.py —— 保护规则 + 代价函数
split_outputs = {m.cv1.conv for m in model.modules() if isinstance(m, (C2f, C2PSA))}
for dep, idxs in group:
    module_name = reverse[dep.target.module]
    out_prune = graph.is_out_channel_pruning_fn(dep.handler)
    if module in split_outputs and out_prune:
        raise ValueError(f"protect_CSP_chunk_width: {module_name}")
    if module_name == "model.0" or module_name.startswith(("model.0.", "model.10.")):
        raise ValueError(f"protect_stem_or_attention: {module_name}")
    if module_name.startswith("model.23.dfl"):
        raise ValueError(f"protect_DFL: {module_name}")

# 贪心的「代价」：每省 1% GMAC 掉多少 mAP（越小越好）
score = drop / gain  # drop = 当前步 mAP 下降；gain = 相对 baseline 的 GMAC 减少

In [ ]:
# 来源：scripts/greedy_evaluation.py —— 微调恢复：直塞模型 + 固定 BN
from ultralytics.models.yolo.detect.train import DetectionTrainer


class FrozenBNTrainer(DetectionTrainer):
    def _model_train(self):
        super()._model_train()  # 先让父类把模型置为 train()
        for m in self.model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()  # 固定 BN：小数据微调别让统计量被冲坏
                for p in m.parameters():
                    p.requires_grad_(False)


trainer = FrozenBNTrainer(overrides=overrides)
trainer.model = pruned.model  # 关键：直接塞已剪模型，绕过 get_model() 按 YAML 重建
trainer.train()

### 梯度分档剪枝（实验 06）

**原理**

- **梯度重要性（Taylor 类）**：`mean(|W × ∂L/∂W|)`——权重 × 权重梯度 的绝对值。直观理解：这个通道对损失函数的贡献/敏感度，比纯 L1 幅值更能反映「剪掉后损失会怎么变」。
- **分档（tiered）**：复用实验 03 的 L1 masking 敏感度，把层分成 low(≤0.005) / medium(0.005~0.020] / high(>0.020) 三档，A–E 五组按不同比例剪（如 A = 低 12.5%/中 0/高 0，E = 低 25%/中 15%/高 5%）。
- **collect_scores 只统计不改权重**：过一遍全部训练图算梯度打分，但**不更新权重、固定 BN**，并验证模型张量前后一致（fingerprint 相同）。

**重要 bash 命令**

```bash
python scripts/prune_gradient_tiered.py --device 0 --epochs 10 --max-map-drop 0.02
```

In [ ]:
# 来源：scripts/prune_gradient_tiered.py —— 梯度打分核心
loss, _ = model(batch)
loss = loss.sum() / size
loss.backward()
for name, total in scores.items():
    weight = modules[name].weight
    value = (
        (weight.detach() * weight.grad.detach()).abs().flatten(1).mean(1)
    )  # |W × ∇L|
    total.add_(value.double().cpu(), alpha=size)  # 累加，最后除以总图数取平均

### 多方法对比：重要性准则（实验 11 coco128）

**原理**

- **目的**：同一基线（exp02 best.pt）、同样 9 个安全层、同样约 10%/层 的剪幅下，横向对比 6 种重要性准则。**控制变量后，差异才纯来自打分函数本身**。
- **安全层（SAFE_LAYERS）**：实验 03 敏感度验证过的 9 个「剪了不疼」的层（低敏感 + 输出通道 ≥ 64 + 排除首层/注意力/检测头 + 排除 TP-1.6.1 残差输出 bug 层 model.13.m.0.cv2.conv）。YOLO11n 冗余小，总剪幅被锁在 ~3.6% 参数附近——这是所有方法「提升不明显」的根本原因，不是方法不行。
- **统一框架**：所有准则都在同一批候选层上给每个**输出通道**打重要性分，剪掉分数最低的一批；换打分函数 = 换方法。

| 准则 | 原理 |
|---|---|
| L1 | 卷积核绝对值均值小 → 不重要 |
| L2 | 卷积核 L2 范数小 → 不重要 |
| BN-γ | 该 conv 跟随的 BN 缩放因子 \|γ\| 小 → 不重要 |
| FPGM | 离同类通道的几何中位数近 → 冗余（可被替代） |
| Taylor | mean\|W × dL/dW\| 小 → 该通道对损失不敏感 |
| HRank | 输出特征图矩阵秩低 → 信息少 |

**结论：Taylor > FPGM > L1 ≈ L2 ≫ BN-γ**（raw：0.336 / 0.326 / 0.314 / 0.315 / 0.012）。「看梯度敏感度」比「看幅值」更准。BN-γ **不先做稀疏训练直接剪是灾难**——BN 的 γ 本身不稀疏，按它排序 ≈ 乱剪。

**名词**：几何中位数 = 与所有同类通道距离和最小的点；HRank 的秩 = 特征矩阵 SVD 后有效奇异值个数。

**重要 bash 命令**

```bash
# 实验 11 脚本在临时目录（不在仓库 scripts/ 下），全部在 coco128 上跑
cd C:/Users/22565/AppData/Local/Temp/yolo_coco128_impr
python e0_heldout.py              # 先用 held-out val2017 重评历史模型（校准评测口径）
python e2_criteria.py             # 5 种准则一次性剪 ~10%/层 + 50ep 微调 + 双口径评测
python e5_sfp_hrank.py hrank      # HRank：hook 收集特征 → SVD 秩打分 → 剪 + 微调
```

In [ ]:
# 来源：yolo_coco128_impr/pruner.py —— 同一批安全层，换打分函数即换方法
def score_l1(model):  # L1：卷积核绝对值均值，越小越不重要
    scores = {}
    for name, m in candidate_layers(model):
        w = m.weight.detach()
        scores[name] = w.abs().mean(dim=(1, 2, 3)).cpu().numpy()
    return scores


def score_l2(model):  # L2：卷积核 L2 范数
    scores = {}
    for name, m in candidate_layers(model):
        w = m.weight.detach()
        scores[name] = w.pow(2).sum(dim=(1, 2, 3)).sqrt().cpu().numpy()
    return scores


def score_bn(model):  # BN-γ：该 conv 跟随的 BN 缩放因子 |γ|（找不到 BN 退回 L1）
    scores = {}
    modules = dict(model.named_modules())
    for name, m in candidate_layers(model):
        w = m.weight.detach()
        parent = modules.get(name.rsplit(".", 1)[0])
        bn = getattr(parent, "bn", None)
        gamma = bn.weight.detach() if isinstance(bn, nn.BatchNorm2d) else None
        scores[name] = (
            gamma.abs() if gamma is not None else w.abs().mean(dim=(1, 2, 3))
        ).cpu().numpy()
    return scores


def score_fpgm(model):  # FPGM：到其余通道的成对距离和（离几何中位数近 → 冗余）
    scores = {}
    for name, m in candidate_layers(model):
        w = m.weight.detach().float().reshape(m.weight.shape[0], -1)  # (C, K)
        wn = (w**2).sum(1, keepdim=True)
        d2 = (wn + wn.t() - 2 * w @ w.t()).clamp(min=0)  # 成对欧氏距离平方
        scores[name] = d2.sqrt().sum(dim=1).cpu().numpy()  # 和越大越不可替代
    return scores


def score_taylor(model, split_dir, n_batches=13):  # Taylor：mean|W × dL/dW|
    # 模型 train 但 BN 固定 eval（统计量别被冲坏）；只收集梯度，不更新权重
    acc = {name: torch.zeros(m.out_channels, device="cpu") for name, m in candidate_layers(model)}
    for imgs in TrainImageBatcher(split_dir):
        model.zero_grad(set_to_none=True)
        preds = model(imgs.to(DEVICE))
        loss = sum(t.float().square().mean() for t in tensors(preds))  # 无标签：预测张量平方和当代理损失
        loss.backward()
        with torch.no_grad():
            for name, m in candidate_layers(model):
                if m.weight.grad is not None:
                    acc[name] += (m.weight.detach() * m.weight.grad.detach()).abs().mean(dim=(1, 2, 3)).cpu()
    return {k: (v / n_batches).numpy() for k, v in acc.items()}

In [ ]:
# 来源：yolo_coco128_impr/e2_criteria.py —— 控制变量：同基线 / 同层 / 同剪幅，只换打分函数
model = fresh_base(base)  # 每次都从同一个 best.pt 重载
scores = {
    "l1": score_l1(model),
    "l2": score_l2(model),
    "bn": score_bn(model),
    "fpgm": score_fpgm(model),
}
scores["taylor"] = score_taylor(model, split_dir, n_batches=13)  # 梯度在 13 个真实 batch 上算

for crit in ["l1", "l2", "bn", "fpgm", "taylor"]:
    m = fresh_base(base)
    plan = uniform_plan(m, scores[crit], ratio=0.10)  # 同样 10%/层 的预算
    m, done, skipped = apply_plan(m, plan)            # 依赖图联动真剪（见独立法）
    ckpt = save_pruned(m, SCRATCH / f"e2_{crit}_pruned.pt", base)
    val_map(ckpt, f"e2_{crit}_raw2017", data=COCO2017_VAL_YAML)   # 剪完先测 raw
    best = finetune(ckpt, f"e2_ft_{crit}", epochs=50, lr0=0.001)  # 统一 50ep 微调
    val_map(best, f"e2_{crit}_val2017", data=COCO2017_VAL_YAML)   # 微调后再测

In [ ]:
# 来源：yolo_coco128_impr/e5_sfp_hrank.py —— HRank：输出特征图秩低 → 先剪
K = 512  # 每个通道特征图采样 K 个空间位置
feats = {name: [] for name, _ in cands}  # 累积成 (C, N_total, K)


def make_hook(name):
    def hook(module, inp, out):
        x = out.detach().float()
        if x.dim() != 4:
            return
        B, C, H, W = x.shape
        xf = x.permute(1, 0, 2, 3).reshape(C, B, H * W)  # (C, B, HW)
        idx = torch.linspace(0, H * W - 1, K, dtype=torch.long)
        feats[name].append(xf[:, :, idx])  # 只留 K 个采样位置
    return hook


for name, _ in cands:  # 每个候选层挂前向 hook，过一遍全部训练图收集特征
    handles.append(md[name].register_forward_hook(make_hook(name)))

# 每个通道的 (N, K) 特征矩阵做 SVD：有效奇异值个数 = 秩，低 → 信息少 → 分数低
mat = torch.cat(feats[name], dim=1)  # (C, N_total, K)
sv = torch.linalg.svdvals(mat[c])
ranks[c] = (sv > 0.9 * sv[0]).sum().item()
# 之后与普通准则完全一致：uniform_plan(ratio=0.10) → apply_plan → finetune

### 恢复手段对比（实验 11 续 coco128）

**原理**

剪完精度必然掉，各手段的思路是「怎么把精度补回来」：

| 手段 | 原理 | 结果（held-out val2017） |
|---|---|---|
| 微调 | 剪完继续训 50ep，让模型适应新结构 | 标准做法，L1 raw 0.314 → 0.340 |
| 软剪枝 SFP | 先**归零**通道（结构还在）训练几轮让模型适应，再一次性硬剪 | **本轮最佳 0.357**（剪幅最小 2.6%） |
| Network Slimming | 先给 BN-γ 加 L1 惩罚稀疏训练 150ep，再按 \|γ\| 剪 | 对 YOLO11n **失败**（γ 压不稀疏，raw 0.004） |
| KD 蒸馏 | 剪后学生向未剪教师的输出 logits 对齐 | 无增益（0.333 vs 普通微调 0.340） |
| 迭代剪枝 | 3 轮 {剪 ~4%/层 → 微调 30ep} | 0.329，不比一次性剪好 |
| 量化 INT8 | 权重/激活转低精度 | 未跑（缺 onnx/tensorrt） |

**结果速查**（YOLO11n，held-out val2017，基准 0.386）：

| 方法 | raw | 微调后 | 参数降幅 |
|---|---:|---:|---:|
| L1 | 0.314 | 0.340 | 3.7% |
| L2 | 0.315 | 0.340 | 3.7% |
| FPGM | 0.326 | 0.337 | 3.7% |
| **Taylor** | **0.336** | **0.344** | 3.7% |
| BN-γ（无稀疏） | 0.012 | 0.195 | 3.7% |
| Network Slimming | 0.004 | 0.096 | 3.7% |
| 迭代剪枝 | — | 0.329 | 4.0% |
| **SFP** | — | **0.357** | 2.6% |
| HRank | — | 0.319 | 3.7% |
| KD 蒸馏 | — | 0.333 | 3.7% |

**评测口径（重要教训）**：必须用 **held-out val2017**（coco2017 的 5000 张验证集）。coco128 的 26-val 会系统性高估 ~0.24 且**低估剪枝损失**——仓库早先「independent > greedy」的结论就是被这个口径污染（E0 重评发现）。

**后续方向（按杠杆排序）**：① 量化 INT8（体积 −75%、速度 2~3×、几乎不掉点，最大免费杠杆）；② 对 C2f split/concat 做成对结构化剪枝（剪幅可从 3.6% 提到 30%+）；③ 高剪枝率 + COCO2017 长微调；④ 换更大的 YOLO11s/m（冗余多、剪枝收益更明显）。

**重要 bash 命令**

```bash
cd C:/Users/22565/AppData/Local/Temp/yolo_coco128_impr
python e5_sfp_hrank.py sfp   # SFP 软剪枝：5 轮 {归零 2%/层 → 微调 4ep} → 硬剪 → 50ep 微调
python e3_sparse.py          # Network Slimming：BN-γ L1 稀疏训练 150ep → 按 |γ| 剪 → 微调
python e4_iterative.py       # 迭代剪枝：3 轮 {剪 ~4%/层 → 微调 30ep}
python e6_kd.py              # KD 蒸馏：L1 剪后学生 + 未剪教师，50ep
python e7_quant.py           # FP16/INT8 导出 + 测速（需 SCRATCH_PKGS 里的 onnx 包）
```

In [ ]:
# 来源：yolo_coco128_impr/e5_sfp_hrank.py —— SFP 软剪枝：先归零让模型适应，再硬剪
masks = {}  # name -> 累计要归零的通道集合
for rnd in range(5):
    scores = score_l2(m)
    for name, _ in candidate_layers(m):
        alive = [i for i in range(len(scores[name])) if i not in masks.get(name, set())]
        cut = max(1, int(len(alive) * 0.02))  # 每轮只归零 2% 存活通道（剪幅最小 → 掉点最少）
        order = torch.argsort(torch.as_tensor(scores[name], dtype=torch.float32))
        pick = [i for i in order.tolist() if i in alive][:cut]
        masks.setdefault(name, set()).update(pick)
        dict(m.named_modules())[name].weight[pick] = 0.0  # 归零（结构还在，参数还在）

    def rezero(trainer):  # 关键：每个 train batch 结束再归零，防优化器把权重写回来
        with torch.no_grad():
            md = dict(trainer.model.named_modules())
            for name, idxs in masks.items():
                if name in md:
                    md[name].weight[list(idxs)] = 0.0

    trainer = DetectionTrainer(overrides=overrides)
    trainer.model = m  # 直接塞模型，防结构重建
    trainer.callbacks["on_train_batch_end"].append(rezero)
    trainer.train()  # 微调 4ep，让网络适应「这些通道已死」
    m = YOLO(str(trainer.best)).model.float().cuda()  # 重载 best，去掉优化器动量伪影

# 5 轮软剪结束：把归零通道一次性硬剪（依赖图联动），再正常 50ep 微调
m2, done, skipped = apply_plan(fresh_base(base), {n: sorted(v) for n, v in masks.items() if v})
best = finetune(m2, "e5_ft_sfp", epochs=50, lr0=0.001)  # 本轮最佳：held-out 0.357

In [ ]:
# 来源：yolo_coco128_impr/e3_sparse.py —— Network Slimming：稀疏化 BN-γ，再按 |γ| 剪
class SparseCriterion:
    """在检测损失上加 sr * sum(|γ|)，逼 BN-γ 稀疏（骨干/颈部全部 BN，排除检测头 model.23）"""

    def __call__(self, preds, batch):
        loss, items = self.base(preds, batch)
        pen = torch.zeros((), device=loss.device, dtype=loss.dtype)
        for m in self.bns:
            pen = pen + m.weight.abs().sum()
        return loss + self.sr * pen, items


model.model.criterion = SparseCriterion(model.model.criterion, model.model, sr=0.1)
trainer.train()  # 稀疏训练 150ep
sparse_last = Path(trainer.last).resolve()  # 关键：用 last.pt 剪——best 是早期还没稀疏的
plan = uniform_plan(fresh_base(sparse_last), score_bn(sparse_last), ratio=0.10)  # 按 |γ| 剪
# 结论：YOLO11n 上 γ 压不稀疏（惩罚与检测损失打架），剪完灾难（raw 0.004），此路对该模型失败

In [ ]:
# 来源：yolo_coco128_impr/e6_kd.py —— KD 蒸馏：剪后学生向未剪教师对齐
class KDCriterion:
    """base 检测损失 + lam * MSE(学生原始 logits, 教师原始 logits)。
    检测头被保护规则保留，师生输出形状完全一致，KD 才能直接做 MSE。"""

    def __call__(self, preds, batch):
        loss, items = self.base(preds, batch)
        img = batch["img"]
        with torch.no_grad():
            self.head.train()  # 强制教师 Detect 头走 train 分支，输出与学生同形状的原始 logits
            t_preds = self.teacher.predict(img)  # 走正确的 skip-connection 路由
            self.head.eval()
        s, t = self._flat(preds), self._flat(t_preds)
        kd = torch.zeros((), device=img.device)
        n = 0
        for a, b in zip(s, t):
            if a.shape == b.shape:
                kd = kd + F.mse_loss(a.float(), b.float())
                n += 1
        if n:
            kd = kd / n
        return loss + self.lam * kd, items

# 教师 eval + 冻结；学生是 E2 的 L1 剪后 checkpoint（最差恢复场景）
# 结论：本轮无增益（0.333 vs 普通微调 0.340）——coco128 太小，教师能教的有限

### 测速与计算量统计（GMACs / 参数 / 前向耗时）

**原理**

- **GMACs / 参数**：用 torch-pruning 的 `count_ops_and_params` 在**未融合(unfused)**结构上统计，保证剪枝前后口径一致。
- **测速**：`fuse()` 融合 Conv+BN（推理常用、更快），batch=1、640×640，预热 30 次后测量 250 次取**中位数**（抗偶发波动）。用 CUDA Event 计时 + 同步。
- **为什么「FLOPs 降了但没加速」**：减的量太少（~1–4%）时，轮间波动比收益还大；真实推理还受内存带宽/算子调度影响。

**名词**：fuse（融合）= 把 Conv 后的 BN 合并进 Conv 权重，推理时少一层计算。

**重要 bash 命令**

```bash
# 测速和 GMACs 已内嵌在剪枝脚本里（benchmark / stats），一般无需单独跑
```

In [ ]:
# 来源：scripts/greedy_evaluation.py —— 测速核心：融合 + 预热 + CUDA Event + 中位数
model = YOLO(path).model.float().eval().fuse()  # 融合 Conv+BN
example = torch.zeros(1, 3, 640, 640, device=device)
with torch.inference_mode():
    for _ in range(30):  # 预热 30 次
        model(example)
    torch.cuda.synchronize()
    start, end = torch.cuda.Event(True), torch.cuda.Event(True)
    start.record()
    model(example)
    end.record()
    torch.cuda.synchronize()
    elapsed = start.elapsed_time(end)  # 单次前向耗时(ms)
# 多轮取 250 次的中位数作为 latency_median_ms